In [1]:
# ==============================================================================
# CELL 1: ENVIRONMENT SETUP & IMPORTS
# ==============================================================================
# Install required libraries if not already present
!pip install -q mne moabb tqdm

import os
import copy
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import scipy.signal as signal
from scipy.linalg import eigh
from sklearn.linear_model import Lasso
from sklearn.metrics import confusion_matrix
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from tqdm.auto import tqdm

# Ensure reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {device}")

# Utility: ShapeTracker for debugging tensor transformations
class ShapeTracker(nn.Module):
    def __init__(self, name=""):
        super().__init__()
        self.name = name

    def forward(self, x, debug=False):
        if debug:
            print(f"[ShapeTracker] {self.name}: {tuple(x.shape)}")
        return x

Using Device: cpu


In [2]:
# ==============================================================================
# CELL 2: ASCII ARCHITECTURE INFOGRAPHICS
# ==============================================================================
def print_architecture_diagrams():
    diagrams = """
    ===========================================================================
    1. PREPROCESSING & FBCSP + LASSO PIPELINE
    ===========================================================================
    Raw EEG (C=22, T=1000) -> NaN Fill -> 5th-order Butterworth (10 Bands)
        |-> Band 1: 1-4 Hz   ...   Band 10: 35-38 Hz
        |-> Z-Score Norm (fit on train)
        |-> OVR-CSP (4 classes x 4 filters = 16 per band -> 160 total)
        |-> LASSO L1-penalized regression -> Selects 'Var' non-zero features
        \\=> Output Z = W_csp^T * X'  [Shape: (Batch, Var, 1000)]

    ===========================================================================
    2. FBGAN GENERATOR (G_theta)
    ===========================================================================
    [Input] Noise z ~ N(0,1)  shape: (Batch, 1600)
       |
    [FC Layer] 1600 -> 256,000 -> Reshape to (Batch, 128, 40, 50)
       |
    [ConvTrans 1 & 2] K=(3,15), S=(1,3) | 128 -> 128 channels
       |
    [ConvTrans 3] K=(3,5), S=(1,2)      | 128 -> 64 channels
       |
    [ConvTrans 4] K=(4,5), S=(2,1)      | 64 -> 32 channels
       |
    [ConvTrans 5] K=(1,2), S=(1,1)      | 32 -> 1 channel
       |
    [Interpolate] Output forces strict Shape: (Batch, 1, 22, 1000)

    ===========================================================================
    3. CRNN-DF CLASSIFIER & CENTER LOSS
    ===========================================================================
    [Input] EEG Signal (Batch, 1, 22, 1000)
       |
    [Conv2D] K=(22, 45) -> Learns spatial/temporal correlation
       |
    [MaxPool2D] K=(1, 75), S=(1, 10)
       |
    [LSTM Module] 2 Layers, hidden=64 -> Captures Temporal Dynamics
       |
    [Linear] -> Output (4 Classes)
       |
    [Loss] L = CrossEntropy + 0.1 * L_cen (Central Distance + Centroid Shift)
    ===========================================================================
    """
    print(diagrams)

print_architecture_diagrams()


    1. PREPROCESSING & FBCSP + LASSO PIPELINE
    Raw EEG (C=22, T=1000) -> NaN Fill -> 5th-order Butterworth (10 Bands)
        |-> Band 1: 1-4 Hz   ...   Band 10: 35-38 Hz
        |-> Z-Score Norm (fit on train)
        |-> OVR-CSP (4 classes x 4 filters = 16 per band -> 160 total)
        |-> LASSO L1-penalized regression -> Selects 'Var' non-zero features
        \=> Output Z = W_csp^T * X'  [Shape: (Batch, Var, 1000)]

    2. FBGAN GENERATOR (G_theta)
    [Input] Noise z ~ N(0,1)  shape: (Batch, 1600)
       |
    [FC Layer] 1600 -> 256,000 -> Reshape to (Batch, 128, 40, 50)
       |
    [ConvTrans 1 & 2] K=(3,15), S=(1,3) | 128 -> 128 channels
       |
    [ConvTrans 3] K=(3,5), S=(1,2)      | 128 -> 64 channels
       |
    [ConvTrans 4] K=(4,5), S=(2,1)      | 64 -> 32 channels
       |
    [ConvTrans 5] K=(1,2), S=(1,1)      | 32 -> 1 channel
       |
    [Interpolate] Output forces strict Shape: (Batch, 1, 22, 1000)

    3. CRNN-DF CLASSIFIER & CENTER LOSS
    [Input] EEG Si

In [3]:
# ==============================================================================
# CELL 3: REAL BCI IV-2a DATASET LOADER (STRICT SHAPE MATCHER)
# ==============================================================================
try:
    import moabb
    from moabb.datasets import BNCI2014001
    from moabb.paradigms import MotorImagery
    moabb.set_log_level('WARNING')
except ImportError:
    raise ImportError("Please ensure mne and moabb are installed.")

def load_real_bci_iv_2a(subjects=[1, 2]):
    """
    Downloads and formats the official BCI Competition IV 2a dataset.
    Channels = 22 EEG, Timepoints = strictly 1000 (4.0s at 250Hz).
    """
    print(f"Fetching BCI Comp IV 2a for Subjects {subjects}. This may take a moment to download...")
    
    dataset = BNCI2014001()
    paradigm = MotorImagery(n_classes=4, resample=250, tmin=0.0, tmax=4.0)
    
    X, labels, metadata = paradigm.get_data(dataset=dataset, subjects=subjects)
    
    # Map string labels to integer classes
    label_map = {'left_hand': 0, 'right_hand': 1, 'feet': 2, 'tongue': 3}
    y = np.array([label_map[l] for l in labels])
    
    n_subjects = len(subjects)
    trials_per_sub = len(X) // n_subjects
    
    X_all = X.reshape(n_subjects, trials_per_sub, X.shape[1], X.shape[2])
    y_all = y.reshape(n_subjects, trials_per_sub)
    
    # Slice off EOGs to keep exactly 22 channels and truncate to exactly 1000 timepoints
    X_all = X_all[:, :, :22, :1000] 
    
    print(f"Real Data Loaded Successfully!")
    print(f"Total Shape - X: {X_all.shape}, y: {y_all.shape}")
    print(f"Class distribution per subject: {np.bincount(y_all[0])}")
    
    return X_all, y_all

# Using Subjects 1 and 2 for reasonable execution time. Expand list to 1-9 for full paper scale.
X_all, y_all = load_real_bci_iv_2a(subjects=[1, 2])

2026-08-21 08:37:12,262 WARNING MainThread moabb.utils BNCI2014001 has been renamed to BNCI2014_001. BNCI2014001 will be removed in version 1.1.
2026-08-21 08:37:12,263 WARNING MainThread moabb.datasets.base The dataset class name 'BNCI2014001' must be an abbreviation of its code 'BNCI2014-001'. See moabb.datasets.base.is_abbrev for more information.
2026-08-21 08:37:12,263 WARNING MainThread moabb.paradigms.motor_imagery Choosing from all possible events


Fetching BCI Comp IV 2a for Subjects [1, 2]. This may take a moment to download...
Real Data Loaded Successfully!
Total Shape - X: (2, 576, 22, 1000), y: (2, 576)
Class distribution per subject: [144 144 144 144]


In [4]:
# ==============================================================================
# CELL 4: PREPROCESSING PIPELINE (FILTER BANK & Z-SCORE)
# ==============================================================================
class EEGPreprocessor:
    def __init__(self, fs=250):
        self.fs = fs
        self.bands = [
            (1, 4), (4, 8), (8, 12), (12, 16), (16, 20),
            (20, 24), (24, 28), (28, 32), (32, 35), (35, 38)
        ]
        self.mean = None
        self.var = None

    def clean_nans(self, X):
        mask = np.isnan(X)
        if mask.any():
            X[mask] = np.nanmean(X)
        return X

    def bandpass_filter(self, data, lowcut, highcut, order=5):
        nyq = 0.5 * self.fs
        low = lowcut / nyq
        high = highcut / nyq
        b, a = signal.butter(order, [low, high], btype='band')
        return signal.filtfilt(b, a, data, axis=-1)

    def fit_zscore(self, X):
        self.mean = np.mean(X, axis=(0, -1), keepdims=True)
        self.var = np.var(X, axis=(0, -1), keepdims=True) + 1e-8

    def apply_zscore(self, X):
        return (X - self.mean) / self.var 

    def process(self, X, is_train=False):
        X = self.clean_nans(X)
        
        X_bands = []
        for (low, high) in self.bands:
            X_b = self.bandpass_filter(X, low, high)
            X_bands.append(X_b)
        X_fb = np.stack(X_bands, axis=1) 
        
        X_composite = np.sum(X_fb, axis=1) 
        
        if is_train:
            self.fit_zscore(X_composite)
            
        X_norm = self.apply_zscore(X_composite)
        return X_fb, X_norm

In [5]:
# ==============================================================================
# CELL 5: FBCSP & LASSO FEATURE SELECTION (WITH SCALING)
# ==============================================================================
from sklearn.preprocessing import StandardScaler

class FBCSPLasso:
    def __init__(self, m_filters=4, lambda_lasso=0.05):
        self.m_filters = m_filters 
        self.lambda_lasso = lambda_lasso
        self.W_csp = None 
        self.Var = 0 
        self.scaler = StandardScaler()
        
    def _compute_cov(self, X):
        cov = X @ X.T
        return cov / np.trace(cov)

    def fit(self, X_fb, y):
        Batch, Bands, C, T = X_fb.shape
        classes = np.unique(y)
        all_filters = []
        
        for b in range(Bands):
            X_b = X_fb[:, b, :, :]
            band_filters = []
            
            for c in classes:
                X_c = X_b[y == c]
                X_rest = X_b[y != c]
                
                R_c = np.mean([self._compute_cov(x) for x in X_c], axis=0)
                R_rest = np.mean([self._compute_cov(x) for x in X_rest], axis=0)
                
                vals, vecs = eigh(R_c, R_rest)
                idx = np.argsort(vals)[::-1]
                vecs = vecs[:, idx]
                
                band_filters.append(vecs[:, :self.m_filters])
            
            all_filters.append(np.hstack(band_filters))
            
        W_full = np.hstack(all_filters) 
        
        # Calculate variances
        X_flat = np.sum(X_fb, axis=1) 
        Z_full = np.array([W_full.T @ x for x in X_flat]) 
        features = np.var(Z_full, axis=2) 
        
        # NEW: Standardize features before applying L1 penalty to ensure consistent sparsity
        features_scaled = self.scaler.fit_transform(features)
        
        lasso = Lasso(alpha=self.lambda_lasso, max_iter=10000)
        lasso.fit(features_scaled, y)
        
        selected_indices = np.where(lasso.coef_ != 0)[0]
        if len(selected_indices) == 0:
            print("Warning: LASSO too strict. Defaulting to top 16 variance features.")
            selected_indices = np.argsort(np.abs(lasso.coef_))[-16:]
            
        self.W_csp = W_full[:, selected_indices]
        self.Var = self.W_csp.shape[1]
        print(f"FBCSP+LASSO: Reduced 160 features -> {self.Var} sparse filters.")

    def transform(self, X_norm):
        Z = np.array([self.W_csp.T @ x for x in X_norm])
        return Z

In [6]:
# ==============================================================================
# CELL 6: FBGAN (FILTER BANK GAN)
# ==============================================================================
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(1600, 256000)
        self.ct1 = nn.ConvTranspose2d(128, 128, kernel_size=(3, 15), stride=(1, 3))
        self.bn1 = nn.BatchNorm2d(128)
        self.ct2 = nn.ConvTranspose2d(128, 128, kernel_size=(3, 15), stride=(1, 3))
        self.bn2 = nn.BatchNorm2d(128)
        self.ct3 = nn.ConvTranspose2d(128, 64, kernel_size=(3, 5), stride=(1, 2))
        self.bn3 = nn.BatchNorm2d(64)
        self.ct4 = nn.ConvTranspose2d(64, 32, kernel_size=(4, 5), stride=(2, 1))
        self.bn4 = nn.BatchNorm2d(32)
        self.ct5 = nn.ConvTranspose2d(32, 1, kernel_size=(1, 2), stride=(1, 1))
        self.tracker = ShapeTracker("FBGAN_Generator")

    def forward(self, z, debug=False):
        x = self.fc(z)
        x = x.view(-1, 128, 40, 50) 
        x = F.leaky_relu(self.bn1(self.ct1(x)), 0.2)
        x = F.leaky_relu(self.bn2(self.ct2(x)), 0.2)
        x = F.leaky_relu(self.bn3(self.ct3(x)), 0.2)
        x = F.leaky_relu(self.bn4(self.ct4(x)), 0.2)
        x = torch.tanh(self.ct5(x))
        x = F.interpolate(x, size=(22, 1000), mode='bilinear', align_corners=False)
        return self.tracker(x, debug)

class Discriminator_Phi(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=(1, 23), stride=(1, 1))
        self.conv2 = nn.Conv2d(10, 30, kernel_size=(22, 1), stride=(1, 1))
        self.conv3 = nn.Conv2d(30, 30, kernel_size=(1, 17), stride=(1, 1))
        self.pool1 = nn.MaxPool2d(kernel_size=(1, 6), stride=(1, 6))
        self.conv4 = nn.Conv2d(30, 30, kernel_size=(1, 7), stride=(1, 7))
        self.pool2 = nn.MaxPool2d(kernel_size=(1, 6), stride=(1, 6))
        self.adap = nn.AdaptiveAvgPool2d((1, 25)) 
        self.fc = nn.Linear(750, 1)

    def forward(self, x):
        x = F.leaky_relu(self.conv1(x), 0.2)
        x = F.leaky_relu(self.conv2(x), 0.2)
        x = F.leaky_relu(self.conv3(x), 0.2)
        x = self.pool1(x)
        x = F.leaky_relu(self.conv4(x), 0.2)
        x = self.pool2(x)
        x = self.adap(x).view(x.size(0), -1)
        return torch.sigmoid(self.fc(x))

class Discriminator_Psi(nn.Module):
    def __init__(self, var):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=(1, 23), stride=(1, 1))
        self.conv2 = nn.Conv2d(10, 30, kernel_size=(4, 1), stride=(4, 1))
        h_after_conv2 = var // 4 if var >= 4 else 1 
        self.conv3 = nn.Conv2d(30, 30, kernel_size=(h_after_conv2, 1), stride=(1, 1))
        self.conv4 = nn.Conv2d(30, 30, kernel_size=(1, 17), stride=(1, 1))
        self.pool1 = nn.MaxPool2d(kernel_size=(1, 6), stride=(1, 6))
        self.conv5 = nn.Conv2d(30, 30, kernel_size=(1, 7), stride=(1, 1))
        self.pool2 = nn.MaxPool2d(kernel_size=(1, 6), stride=(1, 6))
        self.adap = nn.AdaptiveAvgPool2d((1, 25))
        self.fc = nn.Linear(750, 1)

    def forward(self, x):
        x = F.leaky_relu(self.conv1(x), 0.2)
        if x.shape[2] < 4 or x.shape[2] % 4 != 0:
            pad_size = 4 - (x.shape[2] % 4) if x.shape[2] % 4 != 0 else 4 - x.shape[2]
            x = F.pad(x, (0,0,0,pad_size))
        x = F.leaky_relu(self.conv2(x), 0.2)
        x = F.leaky_relu(self.conv3(x), 0.2)
        x = F.leaky_relu(self.conv4(x), 0.2)
        x = self.pool1(x)
        x = F.leaky_relu(self.conv5(x), 0.2)
        x = self.pool2(x)
        x = self.adap(x).view(x.size(0), -1)
        return torch.sigmoid(self.fc(x))

In [7]:
# ==============================================================================
# CELL 7: CRNN-DF CLASSIFIER (CONV + LSTM)
# ==============================================================================
class CRNNDF(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.conv = nn.Conv2d(1, 45, kernel_size=(22, 45), stride=(1, 1))
        self.bn = nn.BatchNorm2d(45)
        self.dropout1 = nn.Dropout(0.5)
        self.pool = nn.MaxPool2d(kernel_size=(1, 75), stride=(1, 10))
        
        self.lstm1 = nn.LSTM(input_size=45, hidden_size=64, batch_first=True)
        self.dropout2 = nn.Dropout(0.5)
        self.lstm2 = nn.LSTM(input_size=64, hidden_size=64, batch_first=True)
        self.dropout3 = nn.Dropout(0.5)
        
        self.fc = nn.Linear(64, num_classes)
        self.tracker = ShapeTracker("CRNN-DF")

    def forward(self, x, debug=False):
        x = F.relu(self.bn(self.conv(x)))
        x = self.dropout1(x)
        x = self.pool(x)
        
        x = x.squeeze(2).permute(0, 2, 1)
        
        x, _ = self.lstm1(x)
        x = self.dropout2(x)
        x, _ = self.lstm2(x)
        x = self.dropout3(x)
        
        features = x[:, -1, :] 
        features = self.tracker(features, debug)
        
        out = self.fc(features)
        return out, features

In [8]:
# ==============================================================================
# CELL 8: DISCRIMINATIVE CENTER LOSS & CENTROID SHIFT (VECTORIZED)
# ==============================================================================
class CenterLossWithShift(nn.Module):
    def __init__(self, num_classes=4, feat_dim=64, alpha=0.02, lambda_cen=0.1):
        super().__init__()
        self.num_classes = num_classes
        self.feat_dim = feat_dim
        self.alpha = alpha            
        self.lambda_cen = lambda_cen  
        
        self.register_buffer('centroids', torch.randn(num_classes, feat_dim))
        self.ce_loss = nn.CrossEntropyLoss()

    def update_centroids(self):
        global_center = torch.mean(self.centroids, dim=0, keepdim=True)
        diff = self.centroids - global_center
        norm = torch.norm(diff, p=2, dim=1, keepdim=True) + 1e-8
        self.centroids.add_(self.alpha * (diff / norm))

    def forward(self, features, labels, preds):
        loss_ce = self.ce_loss(preds, labels)
        batch_size = features.size(0)
        centers_batch = self.centroids[labels]
        loss_cen = torch.sum(torch.pow(features - centers_batch, 2)) / batch_size
        
        total_loss = loss_ce + self.lambda_cen * loss_cen
        return total_loss, loss_ce, loss_cen

    def init_centroids(self, dataloader, model):
        model.eval()
        class_feats = {i: [] for i in range(self.num_classes)}
        with torch.no_grad():
            for x, y in dataloader:
                x = x.to(device)
                _, feats = model(x)
                for i, label in enumerate(y):
                    class_feats[label.item()].append(feats[i].detach())
                    
        for j in range(self.num_classes):
            if len(class_feats[j]) > 0:
                self.centroids[j] = torch.stack(class_feats[j]).mean(dim=0).to(self.centroids.device)

In [9]:
# ==============================================================================
# CELL 9: CONVERGENT LOSO TRAINING PIPELINE (OPTIMIZED SCHEDULING)
# ==============================================================================
def train_loso_convergent(X_all, y_all, epochs=100):
    num_subjects = X_all.shape[0]
    test_sub = 0
    print(f"\n--- LOSO: Evaluating on Subject {test_sub+1} ---")
    
    train_idx = [i for i in range(num_subjects) if i != test_sub]
    X_train = X_all[train_idx].reshape(-1, *X_all.shape[2:])
    y_train = y_all[train_idx].flatten()
    X_test = X_all[test_sub]
    y_test = y_all[test_sub]

    print("Running Preprocessing Pipeline...")
    preprocessor = EEGPreprocessor()
    X_train_fb, X_train_norm = preprocessor.process(X_train, is_train=True)
    _, X_test_norm = preprocessor.process(X_test, is_train=False)
    
    print("Fitting FBCSP + LASSO Feature Selection...")
    extractor = FBCSPLasso(m_filters=4, lambda_lasso=0.05)
    extractor.fit(X_train_fb, y_train)
    
    Z_train = extractor.transform(X_train_norm)
    Var = extractor.Var
    W_csp = torch.tensor(extractor.W_csp, dtype=torch.float32).to(device)

    X_train_t = torch.tensor(X_train_norm, dtype=torch.float32).unsqueeze(1).to(device)
    y_train_t = torch.tensor(y_train, dtype=torch.long).to(device)
    Z_train_t = torch.tensor(Z_train, dtype=torch.float32).unsqueeze(1).to(device)
    
    print("Training Parallel Class-Wise FBGANs...")
    fake_samples, fake_labels_list = [], []
    
    for c in range(4):
        c_mask = (y_train_t == c)
        X_c = X_train_t[c_mask]
        Z_c = Z_train_t[c_mask]
        
        G = Generator().to(device)
        D_phi = Discriminator_Phi().to(device)
        D_psi = Discriminator_Psi(Var).to(device)
        
        opt_G = optim.Adam(G.parameters(), lr=0.0002, betas=(0.5, 0.999))
        opt_D = optim.Adam(list(D_phi.parameters()) + list(D_psi.parameters()), lr=0.0002, betas=(0.5, 0.999))
        bce = nn.BCELoss()
        
        # NEW: Increased GAN epochs for proper distribution mapping
        gan_epochs = 40 
        for ep in tqdm(range(gan_epochs), desc=f"FBGAN (Class {c})", leave=False):
            for i in range(0, len(X_c), 5):
                rx = X_c[i:i+5]
                rz = Z_c[i:i+5]
                bsz = rx.size(0)
                if bsz < 2: continue
                
                opt_D.zero_grad()
                z = torch.randn(bsz, 1600).to(device)
                fx = G(z)
                fz = torch.matmul(W_csp.T, fx.squeeze(1)).unsqueeze(1)
                
                loss_D = bce(D_phi(rx), torch.ones(bsz, 1).to(device)) + \
                         bce(D_phi(fx.detach()), torch.zeros(bsz, 1).to(device)) + \
                         bce(D_psi(rz), torch.ones(bsz, 1).to(device)) + \
                         bce(D_psi(fz.detach()), torch.zeros(bsz, 1).to(device))
                loss_D.backward()
                opt_D.step()
                
                opt_G.zero_grad()
                loss_G = bce(D_phi(fx), torch.ones(bsz, 1).to(device)) + \
                         bce(D_psi(fz), torch.ones(bsz, 1).to(device))
                loss_G.backward()
                opt_G.step()
                
        with torch.no_grad():
            num_to_generate = 250
            gen_batch_size = 25
            class_fakes = []
            
            for _ in range(num_to_generate // gen_batch_size):
                z = torch.randn(gen_batch_size, 1600).to(device)
                gen_x = G(z).detach()
                class_fakes.append(gen_x)
                
            class_fakes_tensor = torch.cat(class_fakes, dim=0)
            fake_samples.append(class_fakes_tensor)
            fake_labels_list.append(torch.full((num_to_generate,), c, dtype=torch.long).to(device))
            
        del G, D_phi, D_psi, opt_G, opt_D
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    X_aug = torch.cat([X_train_t] + fake_samples, dim=0)
    y_aug = torch.cat([y_train_t] + fake_labels_list, dim=0)
    
    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_aug, y_aug), 
        batch_size=32, 
        shuffle=True,
        drop_last=True # Prevent batch-norm instability on final uneven batch
    )

    print("Training CRNN-DF Classifier with Discriminative Features...")
    model = CRNNDF(num_classes=4).to(device)
    loss_fn = CenterLossWithShift(num_classes=4, feat_dim=64, alpha=0.02, lambda_cen=0.1).to(device)
    loss_fn.init_centroids(train_loader, model)
    
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-3)
    # NEW: Cosine Annealing to gracefully lower the learning rate, preventing overfitting to GAN noise
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    
    epoch_bar = tqdm(range(1, epochs + 1), desc="CRNN-DF Training")
    for epoch in epoch_bar:
        model.train()
        total_l = 0.0
        for bx, by in train_loader:
            optimizer.zero_grad()
            preds, feats = model(bx)
            loss, l_ce, l_cen = loss_fn(feats, by, preds)
            loss.backward()
            optimizer.step()
            total_l += loss.item()
            
        scheduler.step()
            
        if epoch % 5 == 0:
            loss_fn.update_centroids()
            
        epoch_bar.set_postfix(Loss=f"{total_l/len(train_loader):.4f}", LR=f"{scheduler.get_last_lr()[0]:.6f}")
            
    print("Training Complete.")
    return model, X_test_norm, y_test

model, X_test_norm, y_test = train_loso_convergent(X_all, y_all, epochs=100)


--- LOSO: Evaluating on Subject 1 ---
Running Preprocessing Pipeline...
Fitting FBCSP + LASSO Feature Selection...
FBCSP+LASSO: Reduced 160 features -> 19 sparse filters.
Training Parallel Class-Wise FBGANs...


FBGAN (Class 0):   0%|          | 0/40 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# ==============================================================================
# CELL 10: EVALUATION & VISUALIZATIONS
# ==============================================================================
def evaluate_and_visualize(model, X_test_norm, y_test):
    model.eval()
    X_test_t = torch.tensor(X_test_norm, dtype=torch.float32).unsqueeze(1).to(device)
    y_test_t = torch.tensor(y_test, dtype=torch.long).to(device)
    
    with torch.no_grad():
        preds, features = model(X_test_t)
        predictions = torch.argmax(preds, dim=1).cpu().numpy()
        features_np = features.cpu().numpy()
        
    acc = np.mean(predictions == y_test)
    print(f"Test Accuracy on Held-Out Subject: {acc * 100:.2f}%")
    
    cm = confusion_matrix(y_test, predictions)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # 1. Confusion Matrix
    ax = axes[0]
    cax = ax.matshow(cm, cmap='Blues')
    fig.colorbar(cax, ax=ax)
    for (i, j), z in np.ndenumerate(cm):
        ax.text(j, i, '{:d}'.format(z), ha='center', va='center')
    ax.set_title("Confusion Matrix (4-Class MI)")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks(np.arange(4))
    ax.set_yticks(np.arange(4))
    ax.set_xticklabels(['L Hand', 'R Hand', 'Feet', 'Tongue'])
    ax.set_yticklabels(['L Hand', 'R Hand', 'Feet', 'Tongue'])

    # 2. t-SNE Scatter Plot
    ax2 = axes[1]
    tsne = TSNE(n_components=2, random_state=42, perplexity=30) 
    feats_2d = tsne.fit_transform(features_np)
    scatter = ax2.scatter(feats_2d[:, 0], feats_2d[:, 1], c=y_test, cmap='viridis')
    ax2.set_title("t-SNE Separability of CRNN-DF Features")
    legend1 = ax2.legend(*scatter.legend_elements(), title="Classes")
    ax2.add_artist(legend1)
    
    plt.tight_layout()
    plt.show()

evaluate_and_visualize(model, X_test_norm, y_test)